In [ ]:
# Linalg support
import numpy as np

# Plotting support
import matplotlib.pyplot as plt

# Helper handling
import glob
from dataclasses import dataclass

In [ ]:
@dataclass
class Measurement:
    coupling_strength: float
    state_size: int
    prediction_length: int

    train_losses: list
    test_losses: list
    train_predictions: np.ndarray
    train_targets: np.ndarray
    test_predictions: np.ndarray
    test_targets: np.ndarray
    test_pearson: float
    train_pearson: float

In [ ]:
# coupling stength, state size, prediction length

prediction_lengths =  [1, 10, 20, 50, 100, 500]

results = {}

prefix = "/beegfs/work/stovey/work/PhD/projects/quantum-lab/quantum-reservoir/sine_wave/parameters-scan"
for length in prediction_lengths:
    results[length] = []
    files = glob.glob(f"{prefix}/fit_results/*_{length}.npy")

    for file in files:
        data = np.load(file, allow_pickle=True).item()
        results[length].append(
            [data.coupling_strength, data.state_size, data.test_pearson]
        )

In [ ]:
# Normalize color range for Pearson correlation values across all datasets
norm = plt.Normalize(-1, 1)  # Assuming Pearson values range from -1 to 1
cmap = plt.get_cmap('coolwarm')

plt.rcParams.update({'font.size': 12})

# Create a figure and a 2x3 grid of subplots
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10), constrained_layout=True)

for i, ax in enumerate(axes.flat):
    # Using the same example data for each subplot for demonstration purposes
    # Replace with actual data for each prediction length
    data_points = results[prediction_lengths[i]]
    strength, size, pearson = zip(*data_points)
    scatter = ax.scatter(strength, size, c=pearson, cmap=cmap, norm=norm, s=100)
    
    # Set title for each subplot
    ax.set_title(rf'Prediction Length: {prediction_lengths[i] / 20}$\tau$')
    
    # Only show x and y labels for the outer plots
    if i >= 3:  # Bottom row
        ax.set_xlabel(r'J / $\pi$')
    if i % 3 == 0:  # First column
        ax.set_ylabel('s')

    # Set axis scaling
    ax.set_xscale('log')
    ax.set_yscale('log')

# Creating a single shared colorbar for all subplots
cbar = fig.colorbar(
    scatter, ax=axes.ravel().tolist(), shrink=0.8, label='Pearson Correlation', pad=0.02
    )

plt.savefig('sine-scan.pdf')
plt.show()